In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
# Load  data 
df = pd.read_csv('Country-data.csv')

# Checking for missing values
print("Missing values per column:")
print(df.isnull().sum()) #this will enaables us to see the count of missing values in each column

# Initial look at the data
df.head()

Missing values per column:
country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


Will convert % to actual dollar amount so the algorith knows that 400(5%) is bigger than 20(15%).

For example, if a country has a GDPP of $1,000 and 10% health spending, they only spend $100. If another has a GDPP of $50,000 and 10% health, they spend $5,000. The algorithm needs to know that $5,000 is much bigger than $100.

In [14]:
# Create a copy so we don't overwrite the original data if we need it later
df_cleaned = df.copy()

# Converting %age of GDPP to absolute values
df_cleaned['exports'] = (df_cleaned['exports'] * df_cleaned['gdpp']) / 100
df_cleaned['health']  = (df_cleaned['health']  * df_cleaned['gdpp']) / 100
df_cleaned['imports'] = (df_cleaned['imports'] * df_cleaned['gdpp']) / 100

# Displaying the first few rows to verify the changes
df_cleaned.head()

,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,55.30,41.9174,248.297,1610,9.44,56.2,5.82,553
1,Albania,16.6,1145.20,267.8950,1987.740,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,1712.64,185.9820,1400.440,12900,16.10,76.5,2.89,4460
3,Angola,119.0,2199.19,100.6050,1514.370,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,5551.00,735.6600,7185.800,19100,1.44,76.8,2.13,12200


In [15]:
# Persist cleaned data for Preprocessing.ipynb (Task 2 & 3)
df_cleaned.to_csv("Country-data-cleaned.csv", index=False)
print("Saved: Country-data-cleaned.csv")

Saved: Country-data-cleaned.csv


To ensure mathematical consistency, we performed Feature Engineering by converting the relative percentage indicators for Health, Exports, and Imports into absolute per-capita values.

We also looked if thier were any missig values, however, the dataset was found to be 100% complete, requiring no imputation.

In [18]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Load the data
df = pd.read_csv('Country-data.csv')

# 2. Clean and Create 'df_model'
df_model = df.copy()
df_model['exports'] = (df['exports'] * df['gdpp']) / 100
df_model['health']  = (df['health']  * df['gdpp']) / 100
df_model['imports'] = (df['imports'] * df['gdpp']) / 100

# 3. Run the Clustering engine
feature_cols = ['child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
X = StandardScaler().fit_transform(df_model[feature_cols])
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df_model['Cluster_ID'] = kmeans.labels_

# 4. Generate the Map
fig = px.choropleth(df_model, 
                    locations="country", 
                    locationmode='country names',
                    color="Cluster_ID", 
                    hover_name="country",
                    title="Strategic Aid Allocation: Priority Map",
                    color_continuous_scale='RdYlGn_r') 
fig.show()

/var/folders/tx/c2pgw65x5gnb2z5dz10vzsv40000gn/T/ipykernel_52965/3070191139.py:22: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(df_model,


Cluster 0 (Red - High Priority): Concentrated in Sub-Saharan Africa and parts of South Asia. These countries have the highest Child Mortality and lowest Income. They are the top candidates for immediate emergency aid.

Cluster 1 (Yellow - Developing): Spread across Latin America and Southeast Asia. These are stable nations that require long-term economic and infrastructure support.

Cluster 2 (Green - Developed): Mostly North America and Europe. These are high-income nations with low mortality rates that do not require humanitarian aid.

In [ ]:

fig = px.scatter(df_model, x="income", y="child_mort", 
                 color="Cluster_ID", hover_name="country",
                 title="The Health-Wealth Gap by Cluster")
fig.show()

Confirms Cluster 0 is the high-priority group with the lowest income and highest mortality.

In [23]:
fig = px.box(df_model, x="Cluster_ID", y="gdpp", color="Cluster_ID",
             title="GDP per Capita Distribution by Cluster")
fig.show()

Validates that the model successfully separated countries by wealth without overlap.